<a href="https://colab.research.google.com/github/3702880330101-netizen/Smart-Cost/blob/main/SmartCost_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd

# Reemplaza con tu URL raw del Gist o captura el parámetro si lo configuras dinámicamente
raw_url = "PEGAR_AQUI_EL_RAW_URL_DEL_JSON"

# Descargar datos desde GitHub (Gist)
response = requests.get(raw_url)
data = response.json()

# Extracción de variables de entrada
material = data.get('material', 'N/A')
consumo_estandar = data.get('consumo_estandar', 0)
precio_material = data.get('precio_material', 0)
desperdicio = data.get('desperdicio', 0) # en porcentaje, ej: 5.0
horas_mod = data.get('horas_mod', 0)
tarifa_mod = data.get('tarifa_mod', 0)
cif_variable = data.get('cif_variable', 0)
costos_fijos = data.get('costos_fijos', 0)
volumen = data.get('volumen', 0) # unidades producidas y vendidas
precio_venta = data.get('precio_venta', 0)

# ==========================================
# CÁLCULOS / REGLAS VERIFICABLES (SEGÚN TABLA)
# ==========================================
material_ajustado = consumo_estandar * (1 + (desperdicio / 100))
costo_mp_unitario = material_ajustado * precio_material
costo_mod_unitario = horas_mod * tarifa_mod
costo_variable_unitario = costo_mp_unitario + costo_mod_unitario + cif_variable
costo_total = (costo_variable_unitario * volumen) + costos_fijos
costo_unitario = costo_total / volumen if volumen > 0 else 0
margen_contribucion_unitario = precio_venta - costo_variable_unitario
punto_equilibrio = costos_fijos / margen_contribucion_unitario if margen_contribucion_unitario > 0 else 0
utilidad = (margen_contribucion_unitario * volumen) - costos_fijos

# Presentación organizada en DataFrame de Pandas
resultados = {
    "Indicador o Regla": [
        "Material ajustado",
        "Costo MP unitario",
        "Costo MOD unitario",
        "Costo variable unitario",
        "Costo total",
        "Costo unitario",
        "Margen de contribución unitario",
        "Punto de equilibrio (u.)",
        "Utilidad"
    ],
    "Resultado": [
        f"{material_ajustado:.4f}",
        f"${costo_mp_unitario:.2f}",
        f"${costo_mod_unitario:.2f}",
        f"${costo_variable_unitario:.2f}",
        f"${costo_total:.2f}",
        f"${costo_unitario:.2f}",
        f"${margen_contribucion_unitario:.2f}",
        f"{punto_equilibrio:.2f} unidades",
        f"${utilidad:.2f}"
    ]
}

df_resultados = pd.DataFrame(resultados)

print(f"--- REPORTE DE COSTOS VERIFICABLES PARA: {material} ---")
display(df_resultados)

In [8]:
# Instalar la librería si es necesario (descomenta la siguiente línea)
!pip install openai

import openai
from google.colab import userdata

# 1. Configurar la API Key de forma segura desde los Secrets de Colab (Llave: OPENAI_API_KEY)
openai.api_key = userdata.get('projects/1081920565379')

# 2. Consolidar los resultados exactos calculados por el motor matemático
contexto_numerico = f"""
RESULTADOS CALCULADOS POR EL MOTOR (ESCENARIO ACTUAL):
- Material analizado: {material}
- Volumen de producción/ventas: {volumen} unidades
- Precio de venta unitario: ${precio_venta:.2f}
- Material ajustado: {material_ajustado:.4f}
- Costo MP unitario: ${costo_mp_unitario:.2f}
- Costo MOD unitario: ${costo_mod_unitario:.2f}
- Costo variable unitario: ${costo_variable_unitario:.2f}
- Costo total: ${costo_total:.2f}
- Costo unitario total: ${costo_unitario:.2f}
- Margen de contribución unitario: ${margen_contribucion_unitario:.2f}
- Punto de equilibrio: {punto_equilibrio:.2f} unidades
- Utilidad operativa: ${utilidad:.2f}
"""

# 3. Construir el prompt estructurado con las directrices de la IA
prompt_ia = f"""
Actúa como analista de costos industriales.
Recibirás resultados YA CALCULADOS por un simulador:
{contexto_numerico}

Tu tarea es:
1) Explicar qué indican estos resultados respecto a la sensibilidad de la utilidad, el margen y el punto de equilibrio.
2) Ordenar las variables de mayor a menor impacto cuantitativo según los datos provistos.
3) Proponer hasta 3 acciones concretas y cuantificables para probar en el simulador.
4) Señalar los supuestos y riesgos asociados.

Reglas estrictas que DEBES cumplir:
- No inventes costos, precios ni volúmenes que no estén en los datos provistos.
- No sustituyas las fórmulas matemáticas programadas en el motor.
- Toda propuesta de acción debe indicar explícitamente qué variable modificar y en qué dirección (ej. reducir desperdicio en X%).
- Distingue claramente el resultado calculado de la recomendación analítica.
- No afirmes que una alternativa es óptima sin una comparación numérica directa.
- Haz evidentes los supuestos sobre los cuales descansan tus recomendaciones.

Formato de salida requerido (respetando estrictamente estas secciones):
- Diagnóstico:
- Variables críticas:
- Acciones a simular:
- Riesgos:
"""

# 4. Llamada al modelo de IA
try:
    response = openai.chat.completions.create(
        model="gpt-4o", # O el modelo estándar de tu preferencia
        messages=[{"role": "user", "content": prompt_ia}],
        temperature=0.2 # Baja temperatura para mantener rigor analítico y evitar inventar datos
    )

    # Imprimir el reporte gerencial generado
    print(response.choices[0].message.content)

except Exception as e:
    print(f"Error al conectar con la API de IA: {e}")

SecretNotFoundError: Secret projects/1081920565379 does not exist.